# DX 704 Week 4 Project

This week's project will test the learning speed of linear contextual bandits compared to unoptimized approaches.
You will start with building a preference data set for evaluation, and then implement different variations of LinUCB and visualize how fast they learn the preferences.


The full project description, a template notebook and supporting code are available on GitHub: [Project 4 Materials](https://github.com/bu-cds-dx704/dx704-project-04).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Collect Rating Data

The file "recipes.tsv" in this repository has information about 100 recipes.
Make a new file "ratings.tsv" with two columns, recipe_slug (from recipes.tsv) and rating.
Populate the rating column with values between 0 and 1 where 0 is the worst and 1 is the best.
You can assign these ratings however you want within that range, but try to make it reflect a consistent set of preferences.
These could be your preferences, or a persona of your choosing (e.g. chocolate lover, bacon-obsessed, or sweet tooth).
Make sure that there are at least 10 ratings of zero and at least 10 ratings of one.


Hint: You may find it more convenient to assign raw ratings from 1 to 5 and then remap them as follows.

`ratings["rating"] = (ratings["rating_raw"] - 1) * 0.25`

In [5]:
import pandas as pd

ratings = pd.read_csv("ratings.tsv", sep="\t")

assert list(ratings.columns) == ["recipe_slug", "rating"]
assert len(ratings) == 100
assert ratings["rating"].between(0, 1).all()
assert (ratings["rating"] == 0).sum() >= 10
assert (ratings["rating"] == 1).sum() >= 10

print("Zero ratings:", (ratings["rating"] == 0).sum())
print("One ratings:", (ratings["rating"] == 1).sum())

ratings.head()

Zero ratings: 12
One ratings: 12


,recipe_slug,rating
0,falafel,0.6
1,spamburger,1.0
2,bacon-fried-rice,1.0
3,chicken-fingers,0.8
4,apple-crisp,0.0


Submit "ratings.tsv" in Gradescope.

## Part 2: Construct Model Input

Use your file "ratings.tsv" combined with "recipe-tags.tsv" to create a new file "features.tsv" with a column recipe_slug, a column bias which is hard-coded to one, and a column for each tag that appears in "recipe-tags.tsv".
The tag column in this file should be a 0-1 encoding of the recipe tags for each recipe.
[Pandas reshaping function methods](https://pandas.pydata.org/docs/user_guide/reshaping.html) may be helpful.

The bias column will make later LinUCB calculations easier since it will just be another dimension.

Hint: For later modeling steps, it will be important to have the feature data (inputs) and the rating data (target outputs) in the same order.
It is highly recommended to make sure that "features.tsv" and "ratings.tsv" have the recipe slugs in the same order.

In [6]:
# YOUR CHANGES HERE
ratings = pd.read_csv("ratings.tsv", sep="\t")
recipe_tags = pd.read_csv("recipe-tags.tsv", sep="\t")

tag_features = pd.crosstab(
    recipe_tags["recipe_slug"],
    recipe_tags["recipe_tag"]
).clip(upper=1).reset_index()

features = ratings[["recipe_slug"]].merge(
    tag_features,
    on="recipe_slug",
    how="left",
    validate="one_to_one"
)

tag_columns = features.columns.drop("recipe_slug")
features[tag_columns] = features[tag_columns].fillna(0).astype(int)

features.insert(1, "bias", 1)

assert features["recipe_slug"].tolist() == ratings["recipe_slug"].tolist()
assert features["bias"].eq(1).all()

features.to_csv("features.tsv", sep="\t", index=False)

print(features.shape)
features.head()

(100, 298)


/tmp/ipykernel_8311/4153022192.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features.insert(1, "bias", 1)


,recipe_slug,bias,alfredo,almond,american,appetizer,appetizers,apple,asiancuisine,asparagus,...,udonnoodles,vanilla,vanillaicecream,vegan,vegetables,vegetarian,warm,whippedcream,winter,yeastdough
0,falafel,1,0,0,0,1,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
1,spamburger,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,bacon-fried-rice,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,chicken-fingers,1,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,apple-crisp,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,1,0


Submit "features.tsv" in Gradescope.

## Part 3: Linear Preference Model

Use your feature and rating files to build a ridge regression model with ridge regression's regularization parameter $\alpha$ set to 1.


Hint: If you are using scikit-learn modeling classes, you should use `fit_intercept=False` since that intercept value will be redundant with the bias coefficient.

Hint: The estimate component of the bounds should match the previous estimate, so you should be able to just focus on the variance component of the bounds now.

In [11]:
# YOUR CHANGES HERE
import numpy as np
from sklearn.linear_model import Ridge


features = pd.read_csv("features.tsv", sep="\t")
ratings = pd.read_csv("ratings.tsv", sep="\t")


assert features["recipe_slug"].tolist() == ratings["recipe_slug"].tolist()


X = features.drop(columns="recipe_slug")
y = ratings["rating"]


model = Ridge(alpha=1, fit_intercept=False)
model.fit(X, y)


ratings["estimate"] = model.predict(X)

print("Number of observations:", X.shape[0])
print("Number of model features:", X.shape[1])
print("Model intercept:", model.intercept_)
print("Coefficient count:", len(model.coef_))

ratings.head()


Number of observations: 100
Number of model features: 297
Model intercept: 0.0
Coefficient count: 297


,recipe_slug,rating,estimate
0,falafel,0.6,0.607734
1,spamburger,1.0,0.975978
2,bacon-fried-rice,1.0,0.999167
3,chicken-fingers,0.8,0.789734
4,apple-crisp,0.0,0.105669


Save the coefficients of this model in a file "model.tsv" with columns "recipe_tag" and "coefficient".
Do not add anything for the `intercept_` attribute of a scikit-learn model; this will be covered by the coefficient for the bias column added in part 2.

In [12]:
# YOUR CHANGES HERE

model_output = pd.DataFrame({
    "recipe_tag": X.columns,
    "coefficient": model.coef_
})

model_output.to_csv("model.tsv", sep="\t", index=False)

print(model_output.shape)
model_output.head()

(297, 2)


,recipe_tag,coefficient
0,bias,0.519763
1,alfredo,0.068303
2,almond,0.002378
3,american,-0.104280
4,appetizer,0.068142


Submit "model.tsv" in Gradescope.

## Part 4: Recipe Estimates

Use the recipe model to estimate the score of every recipe.
Save these estimates to a file "estimates.tsv" with columns recipe_slug and score_estimate.

In [13]:
# YOUR CHANGES HERE

score_estimates = model.predict(X)

estimates = pd.DataFrame({
    "recipe_slug": features["recipe_slug"],
    "score_estimate": score_estimates
})

estimates.to_csv("estimates.tsv", sep="\t", index=False)

print(estimates.shape)
estimates.head()

(100, 2)


,recipe_slug,score_estimate
0,falafel,0.607734
1,spamburger,0.975978
2,bacon-fried-rice,0.999167
3,chicken-fingers,0.789734
4,apple-crisp,0.105669


Submit "estimates.tsv" in Gradescope.

## Part 5: LinUCB Bounds

Calculate the upper bounds of LinUCB using data corresponding to trying every recipe once and receiving the rating in "ratings.tsv" as the reward.
Keep the ridge regression regularization parameter at 1, and set LinUCB's $\alpha$ parameter to 2.
Save these upper bounds to a file "bounds.tsv" with columns recipe_slug and score_bound.

In [14]:
# YOUR CHANGES HERE

features = pd.read_csv("features.tsv", sep="\t")
ratings = pd.read_csv("ratings.tsv", sep="\t")

assert features["recipe_slug"].tolist() == ratings["recipe_slug"].tolist()

X = features.drop(columns="recipe_slug").to_numpy(dtype=float)
y = ratings["rating"].to_numpy(dtype=float)

ridge_alpha = 1.0
linucb_alpha = 2.0

A = ridge_alpha * np.eye(X.shape[1]) + X.T @ X
b = X.T @ y

theta = np.linalg.solve(A, b)

score_estimates = X @ theta

A_solved_XT = np.linalg.solve(A, X.T)
variance_components = np.sum(X.T * A_solved_XT, axis=0)

variance_components = np.maximum(variance_components, 0)

score_bounds = (
    score_estimates
    + linucb_alpha * np.sqrt(variance_components)
)

bounds = pd.DataFrame({
    "recipe_slug": features["recipe_slug"],
    "score_bound": score_bounds
})

bounds.to_csv("bounds.tsv", sep="\t", index=False)

print(bounds.shape)
bounds.head()

(100, 2)


,recipe_slug,score_bound
0,falafel,2.373990
1,spamburger,2.870314
2,bacon-fried-rice,2.865323
3,chicken-fingers,2.583850
4,apple-crisp,1.882011


Submit "bounds.tsv" in Gradescope.

In [17]:
assert list(bounds.columns) == ["recipe_slug", "score_bound"]
assert len(bounds) == 100
assert bounds["recipe_slug"].tolist() == ratings["recipe_slug"].tolist()
assert bounds["score_bound"].notna().all()
assert np.all(score_bounds >= score_estimates - 1e-12)

## Part 6: Make Online Recommendations

Implement LinUCB to make 100 recommendations starting with no data and using the same parameters as in part 5.
One recommendation should be made at a time and you can break ties arbitrarily.
After each recommendation, use the rating from part 1 as the reward to update the LinUCB data.
Record the recommendations made in a file "recommendations.tsv" with columns "recipe_slug", "score_bound", and "reward".
The rows in this file should be in the same order as the recommendations were made.

Hint: do not remove recipes after each recommendation.
Repeating recommendations is expected.

In [15]:
# YOUR CHANGES HERE

features = pd.read_csv("features.tsv", sep="\t")
ratings = pd.read_csv("ratings.tsv", sep="\t")

assert features["recipe_slug"].tolist() == ratings["recipe_slug"].tolist()

recipe_slugs = features["recipe_slug"].to_numpy()
X = features.drop(columns="recipe_slug").to_numpy(dtype=float)
rewards = ratings["rating"].to_numpy(dtype=float)

ridge_alpha = 1.0
linucb_alpha = 2.0
number_of_recommendations = 100

number_of_features = X.shape[1]

A_inverse = np.eye(number_of_features) / ridge_alpha
b = np.zeros(number_of_features)

recommendation_records = []

for step in range(number_of_recommendations):

  
    theta = A_inverse @ b

    
    score_estimates = X @ theta

    
    variances = np.einsum(
        "ij,jk,ik->i",
        X,
        A_inverse,
        X
    )
    variances = np.maximum(variances, 0)

    
    score_bounds = (
        score_estimates
        + linucb_alpha * np.sqrt(variances)
    )

    
    selected_index = np.argmax(score_bounds)

    selected_slug = recipe_slugs[selected_index]
    selected_bound = score_bounds[selected_index]
    observed_reward = rewards[selected_index]
    selected_features = X[selected_index]

    
    recommendation_records.append({
        "recipe_slug": selected_slug,
        "score_bound": selected_bound,
        "reward": observed_reward
    })

    
    A_inverse_x = A_inverse @ selected_features
    denominator = 1.0 + selected_features @ A_inverse_x

    A_inverse = (
        A_inverse
        - np.outer(A_inverse_x, A_inverse_x) / denominator
    )

   
    b += observed_reward * selected_features


recommendations = pd.DataFrame(recommendation_records)

recommendations.to_csv(
    "recommendations.tsv",
    sep="\t",
    index=False
)

print(recommendations.shape)
recommendations.head(10)

(100, 3)


,recipe_slug,score_bound,reward
0,apple-crumble,7.483315,0.0
1,ma-la-chicken,7.192589,0.9
2,quesadillas,7.242484,0.9
3,ramen,7.213720,1.0
4,bacon-fried-rice,7.028568,1.0
5,spamburger,7.044196,1.0
6,chocolate-babka,7.011801,0.5
7,pain-au-chocolat,6.982724,0.6
8,nacho-fries,6.962191,1.0
9,asparagus-burger,6.751829,0.7


Submit "recommendations.tsv" in Gradescope.

In [16]:
assert list(recommendations.columns) == [
    "recipe_slug",
    "score_bound",
    "reward"
]
assert len(recommendations) == 100
assert recommendations["recipe_slug"].isin(recipe_slugs).all()
assert recommendations["reward"].between(0, 1).all()
assert recommendations["score_bound"].notna().all()

print(
    "Unique recipes recommended:",
    recommendations["recipe_slug"].nunique()
)

Unique recipes recommended: 99


## Part 7: Acknowledgments

Make a file "acknowledgments.txt" documenting any outside sources or help on this project.
If you discussed this assignment with anyone, please acknowledge them here.
If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for.
If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy.
If no acknowledgements are appropriate, just write none in the file.


Submit "acknowledgments.txt" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.


Submit "project.ipynb" in Gradescope.